# Module 9 · 圖像 4：資料增強與大規模資料集組織

> **本節定位｜2026 主流核心**
> 把影像變成張量還不夠——訓練視覺模型還需要兩件事：
> (1) **資料增強 (augmentation)** 提升泛化；(2) **可擴展的資料集組織方式**。
> 本節用 `torchvision.transforms.v2` 做增強，並比較資料集的組織格式。

## 學習目標
- 理解資料增強為何能提升泛化，並用 `transforms.v2` 實作常見增強。
- 釐清「訓練」與「驗證/推論」前處理的差異（增強只用於訓練）。
- 認識三種資料集組織方式：**ImageFolder**、**HuggingFace `datasets`**、**WebDataset/Parquet**。

<!-- concept-image:04_augmentation_and_datasets_fig1 -->
<div align="center">
<img src="concept_images/04_augmentation_and_datasets_fig1_augmentation_effects_20260611_221356.png" alt="資料增強效果示意" width="640" style="max-width:100%;">
<br><sub>圖 1 · 資料增強效果示意</sub>
</div>

In [ ]:
import numpy as np
from PIL import Image
from sklearn.datasets import load_sample_images
# 真實照片：flower.jpg
sample = Image.fromarray(load_sample_images().images[1].astype(np.uint8))
print(f"真實影像 flower.jpg, 尺寸 {sample.size}")

## 1. 資料增強：用「合理的隨機變化」擴增訓練分佈

對訓練影像做隨機裁切、翻轉、色彩抖動等，讓模型看到更多樣的樣本、不易過擬合。
**關鍵原則**：增強要符合任務語意（例如手寫數字不能左右翻轉，否則 6 變 9）。

In [ ]:
try:
    import torch
    from torchvision.transforms import v2

    train_aug = v2.Compose([
        v2.RandomResizedCrop(224, antialias=True),   # 隨機裁切+縮放
        v2.RandomHorizontalFlip(p=0.5),              # 隨機水平翻轉
        v2.ColorJitter(brightness=0.2, contrast=0.2),# 色彩抖動
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    # 同一張圖增強兩次 → 得到不同結果（隨機性）
    a, b = train_aug(sample), train_aug(sample)
    print(f"增強輸出形狀: {tuple(a.shape)}  (C, H, W)")
    print(f"兩次增強是否相同: {torch.allclose(a, b)}  → False 代表每次隨機不同")
    HAS_TV = True
except Exception as e:
    HAS_TV = False
    print("未安裝 torchvision，請 `uv sync --extra multimodal`。", e)

## 2. 訓練 vs 驗證：前處理不一樣

**訓練**用隨機增強；**驗證/推論**要**確定性**前處理（固定 resize + center crop），
否則評估結果無法重現。

<!-- concept-image:04_augmentation_and_datasets_fig2 -->
<div align="center">
<img src="concept_images/04_augmentation_and_datasets_fig2_train_vs_val_20260611_183141.png" alt="訓練 vs 驗證前處理的區別" width="640" style="max-width:100%;">
<br><sub>圖 2 · 訓練 vs 驗證前處理的區別</sub>
</div>

In [ ]:
if HAS_TV:
    eval_tf = v2.Compose([
        v2.Resize(256, antialias=True),
        v2.CenterCrop(224),                          # 確定性裁切
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    e1, e2 = eval_tf(sample), eval_tf(sample)
    print(f"驗證前處理兩次是否相同: {torch.allclose(e1, e2)}  → True（確定性）")

## 3. 資料集的組織方式（依規模選擇）

| 方式 | 適用規模 | 結構 | 特點 |
|:--|:--|:--|:--|
| **ImageFolder** | 小~中 | `root/類別名/xxx.jpg` | 最簡單，資料夾名即標籤 |
| **HF `datasets`** | 中~大 | Arrow/Parquet + `Image` 欄 | 可串流、`.map()` 前處理、易分享 |
| **WebDataset / Parquet shards** | 超大（億級） | `.tar` / parquet 分片 | 順序讀取、適合雲端與多機訓練 |

ImageFolder 範例結構：
```
data/
  train/
    cat/  cat001.jpg ...
    dog/  dog001.jpg ...
  val/
    cat/ ...  dog/ ...
```

<!-- concept-image:04_augmentation_and_datasets_fig3 -->
<div align="center">
<img src="concept_images/04_augmentation_and_datasets_fig3_dataset_formats_20260611_221435.png" alt="三種資料集組織方式對比" width="640" style="max-width:100%;">
<br><sub>圖 3 · 三種資料集組織方式對比</sub>
</div>

In [ ]:
# HuggingFace datasets 的 Image 欄位示意（需 multimodal extra）
if HAS_TV:
    try:
        from datasets import Dataset, Image as HFImage
        # 以真實照片建立一個迷你資料集
        ds = Dataset.from_dict({"image": ["/tmp/_a.png"], "label": [0]})
        sample.save("/tmp/_a.png")
        ds = ds.cast_column("image", HFImage())
        print("HF datasets 樣本:", {k: type(v).__name__ for k, v in ds[0].items()})
        print("→ 'image' 欄會在讀取時自動解碼成 PIL.Image；可用 .map() 套上前處理。")
    except Exception as e:
        print("（datasets 示範略過）", e)

## 小結

| 重點 | 內容 |
|:--|:--|
| 訓練增強 | RandomResizedCrop / Flip / ColorJitter…（符合任務語意） |
| 驗證前處理 | 確定性 Resize + CenterCrop + Normalize |
| 小規模 | ImageFolder（資料夾即標籤） |
| 中大規模 | HF `datasets`（可串流、可 map） |
| 超大規模 | WebDataset / Parquet 分片 |

**下一節 `05_dogs_cats_case`**：把整套前處理 + 預訓練模型用在 Dogs vs Cats 影像分類。